
# Bike Sharing Demand Analysis
## DSI Cohort 8 – Infrastructure & Transportation

This notebook follows the task described in the dataset README:
**Regression modeling to predict bike rental demand (`cnt`).**

We demonstrate two regression models:
- Linear Regression (baseline)
- Random Forest Regression

Alongside two advanced visualizations:
1. Demand Heatmap (Hour × Day of Week)
2. Seasonal Rental Distribution (Box Plot)


## 1. Load Libraries and Dataset

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

hour = pd.read_csv("hour_clean (1).csv")
hour.head()


## 2. Visualization 1 — Demand Heatmap (Hour of Day × Temperature Range)

This heatmap shows how bike rental demand changes by **hour of the day** and **temperature range**.
It helps reveal when demand is highest under different weather conditions.


In [ ]:
temp_col = 'temp_celsius' if 'temp_celsius' in hour.columns else 'temp'
if temp_col == 'temp':
    hour['temp_celsius'] = hour['temp'] * 41
    temp_col = 'temp_celsius'

bins = [0, 5, 10, 15, 20, 25, 30, 35, 45]
labels = ['0–5°C', '5–10°C', '10–15°C', '15–20°C', '20–25°C', '25–30°C', '30–35°C', '35–45°C']

hour['temp_bin'] = pd.cut(hour[temp_col], bins=bins, labels=labels, include_lowest=True)

pivot = hour.pivot_table(
    values='cnt',
    index='temp_bin',
    columns='hr',
    aggfunc='mean',
    observed=False
)

plt.figure(figsize=(11, 5))
plt.imshow(pivot, aspect='auto')
plt.title('Bike Rental Demand Heatmap (Temperature Range vs Hour of Day)')
plt.xlabel('Hour of Day')
plt.ylabel('Temperature Range')
plt.xticks(range(len(pivot.columns)), list(pivot.columns))
plt.yticks(range(len(pivot.index)), list(pivot.index))
plt.colorbar(label='Average Rental Count')
plt.tight_layout()
plt.show()



## 3. Visualization 2 — Seasonal Rental Distribution (Box Plot)

This plot shows the **spread, median, and outliers** of bike rental demand across seasons.


In [ ]:

plt.figure()
hour.boxplot(column="cnt", by="season_label")
plt.title("Distribution of Bike Rentals by Season")
plt.suptitle("")
plt.xlabel("Season")
plt.ylabel("Rental Count")
plt.show()



## 4. Prepare Data for Regression Models


In [ ]:

features = [
"hr",
"workingday",
"temp_celsius",
"humidity_pct",
"windspeed_kmh"
]

X = hour[features]
y = hour["cnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



## 5. Model 1 — Linear Regression (Baseline)


In [ ]:

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_r2 = r2_score(y_test, lr_pred)
lr_r2



## 6. Model 2 — Random Forest Regression


In [ ]:

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_r2



## 7. Model Comparison
Higher R² indicates better predictive performance.


In [ ]:

print("Linear Regression R²:", lr_r2)
print("Random Forest R²:", rf_r2)
